In [ ]:
# @title فحص عينات خامات كونكر — الملف الأول فقط
# خلية واحدة: اختيار الأرشيف يدويًا، فحص ZIP والخامات، وإنشاء تقارير قابلة للمراجعة.
from google.colab import drive
from pathlib import Path
from IPython.display import display, clear_output
import ipywidgets as W
import zipfile, hashlib, json, csv, os, io, base64, requests, collections, re
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = False

REPO = "ZzwahwahzZ/Clash-of-Legends"
BRANCH = "forensic/color-texture-sample-audit"
EXPECTED = "Conquer_Color_Texture_Test_3_Samples.zip"
drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive")
archive_exts = {".zip"}
current = ROOT
path_html = W.HTML()
folders = W.Dropdown(options=[], description="المجلد:", layout=W.Layout(width="98%"))
files = W.Select(options=[], rows=10, description="الأرشيف:", layout=W.Layout(width="98%", height="230px"))
up = W.Button(description="⬆ لأعلى")
open_btn = W.Button(description="فتح المجلد")
refresh = W.Button(description="تحديث")
token = W.Password(description="GitHub Token:", placeholder="توكن GitHub بصلاحية Contents write", layout=W.Layout(width="98%"))
run = W.Button(description="فحص الأرشيف المختار", button_style="success", icon="search")
out = W.Output()

def scan(*_):
    global current
    path_html.value = f"<b>المسار:</b> {current}"
    ds, fs = [], []
    try:
        for e in os.scandir(current):
            try:
                if e.is_dir(follow_symlinks=False): ds.append((e.name, e.path))
                elif e.is_file(follow_symlinks=True) and Path(e.name).suffix.lower() in archive_exts:
                    fs.append((f"{e.name} — {e.stat().st_size:,} bytes", e.path))
            except OSError: pass
    except Exception as e:
        with out: print("تعذر قراءة المجلد:", e)
        return
    folders.options = sorted(ds, key=lambda x:x[0].lower())
    files.options = sorted(fs, key=lambda x:x[0].lower())
    files.value = files.options[0][1] if files.options else None

def go_up(_):
    global current
    if current != ROOT and ROOT in current.parents:
        current = current.parent
        scan()

def go_folder(_):
    global current
    if folders.value:
        current = Path(folders.value)
        scan()

def audit(_):
    with out:
        clear_output()
        if not files.value:
            print("اختار أرشيف ZIP أولًا.")
            return
        src = Path(files.value)
        if src.name != EXPECTED:
            print("تنبيه: الملف المختار ليس ملف العينات المحدد.")
            print("المطلوب:", EXPECTED, "| المختار:", src.name)
            return
        if not token.value.strip():
            print("أدخل GitHub Token في الخانة المخصصة.")
            return
        print("الملف:", src.name)
        sha = hashlib.sha256()
        with src.open("rb") as f:
            for block in iter(lambda:f.read(8*1024*1024), b""): sha.update(block)
        archive_sha = sha.hexdigest()
        manifest, image_rows, errors = [], [], []
        ext_counts = collections.Counter()
        total_uncompressed = 0
        try:
            zf = zipfile.ZipFile(src)
            bad = zf.testzip()
            if bad: raise RuntimeError("ZIP CRC failure at: " + bad)
            infos = [i for i in zf.infolist() if not i.is_dir()]
            for info in infos:
                total_uncompressed += info.file_size
                ext = Path(info.filename).suffix.lower() or "[no extension]"
                ext_counts[ext] += 1
                row = {"path":info.filename,"size":info.file_size,"compressed_size":info.compress_size,
                       "extension":ext,"crc32":f"{info.CRC:08x}"}
                manifest.append(row)
                if ext in {".png",".jpg",".jpeg",".bmp",".tga",".webp",".gif",".dds",".ktx",".ktx2"}:
                    try:
                        data = zf.read(info)
                        im = Image.open(io.BytesIO(data))
                        im.load()
                        has_alpha = "A" in im.getbands() or "transparency" in im.info
                        image_rows.append({"path":info.filename,"format":im.format,"width":im.width,
                                           "height":im.height,"mode":im.mode,"has_alpha":bool(has_alpha),
                                           "bytes":info.file_size})
                    except Exception as e:
                        errors.append({"path":info.filename,"error":str(e)})
            zf.close()
        except Exception as e:
            print("فشل فحص الأرشيف:", repr(e))
            return
        summary = {
            "archive":src.name,"archive_bytes":src.stat().st_size,"archive_sha256":archive_sha,
            "zip_crc_test":"PASS","entries":len(manifest),"uncompressed_bytes":total_uncompressed,
            "extension_counts":dict(ext_counts),"images_decoded":len(image_rows),
            "image_decode_errors":len(errors)
        }
        outdir = Path("/content/conquer_color_texture_audit")
        outdir.mkdir(exist_ok=True)
        (outdir/"summary.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding="utf-8")
        (outdir/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
        with (outdir/"textures.csv").open("w",newline="",encoding="utf-8-sig") as f:
            w=csv.DictWriter(f,fieldnames=["path","format","width","height","mode","has_alpha","bytes"])
            w.writeheader(); w.writerows(image_rows)
        with (outdir/"decode_errors.json").open("w",encoding="utf-8") as f:
            json.dump(errors,f,ensure_ascii=False,indent=2)
        # Contact sheet for decoded raster textures only; thumbnails preserve source files unchanged.
        thumbs=[]
        for r in image_rows:
            try:
                raw=zipfile.ZipFile(src).read(r["path"])
                im=Image.open(io.BytesIO(raw)).convert("RGBA")
                im.thumbnail((180,140))
                tile=Image.new("RGB",(210,180),(45,45,45))
                tile.paste(im,((210-im.width)//2,5),im)
                from PIL import ImageDraw
                d=ImageDraw.Draw(tile)
                label=Path(r["path"]).name[:28]
                d.text((6,150),label,fill="white")
                thumbs.append(tile)
            except Exception: pass
        if thumbs:
            cols=4; rows=(len(thumbs)+cols-1)//cols
            sheet=Image.new("RGB",(cols*210,rows*180),(25,25,25))
            for i,t in enumerate(thumbs): sheet.paste(t,((i%cols)*210,(i//cols)*180))
            sheet.save(outdir/"texture_contact_sheet.jpg",quality=88)
        print("نتيجة فحص ZIP:",summary["zip_crc_test"])
        print("عدد الملفات:",len(manifest)," | الحجم بعد فك الضغط:",f'{total_uncompressed:,}',"bytes")
        print("أنواع الملفات:",dict(ext_counts))
        print("صور تم فكها:",len(image_rows)," | أخطاء قراءة الصور:",len(errors))
        print("SHA-256:",archive_sha)
        # Publish only derived audit reports, never the original archive or unrelated Drive files.
        headers={"Authorization":"Bearer "+token.value.strip(),"Accept":"application/vnd.github+json",
                 "X-GitHub-Api-Version":"2022-11-28"}
        check=requests.get(f"https://api.github.com/repos/{REPO}",headers=headers,timeout=30)
        check.raise_for_status()
        if not check.json().get("permissions",{}).get("push",False):
            raise PermissionError("التوكن لا يملك صلاحية الكتابة على المستودع.")
        for p in sorted(outdir.iterdir()):
            if not p.is_file(): continue
            content=base64.b64encode(p.read_bytes()).decode("ascii")
            api=f"https://api.github.com/repos/{REPO}/contents/research/color-texture-sample-audit/{p.name}"
            old=requests.get(api,headers=headers,params={"ref":BRANCH},timeout=30)
            payload={"message":f"Add color texture sample audit: {p.name}","content":content,"branch":BRANCH}
            if old.status_code==200: payload["sha"]=old.json()["sha"]
            elif old.status_code!=404: old.raise_for_status()
            saved=requests.put(api,headers=headers,json=payload,timeout=120)
            saved.raise_for_status()
            print("نُشر التقرير:",p.name)
        token.value=""
        print("\nاكتمل الفحص ونشر التقارير على فرع مستقل:")
        print("https://github.com/"+REPO+"/tree/"+BRANCH+"/research/color-texture-sample-audit")
        print("انسخ هذا التقرير وأرسله هنا. لم يتم تعديل main أو الملفات الأصلية.")

up.on_click(go_up); open_btn.on_click(go_folder); refresh.on_click(scan); run.on_click(audit)
display(W.VBox([W.HTML("<h3>فحص عيّنة خامات الألوان — الملف الأول فقط</h3><p>اختار Conquer_Color_Texture_Test_3_Samples.zip فقط. لا تختار أي ملف آخر.</p>"),
                path_html,W.HBox([up,open_btn,refresh]),folders,files,token,run,out]))
scan()
